# 06 — Sentiment evaluation and calibration

Prepared 9 November 2026 (Asia/Calcutta). **Generated, unexecuted notebook.**

Review actual held-out metrics and explicit language/cohort support without refitting models.

Inputs: actual evaluation JSON produced by ml.sentiment_tfidf.evaluate or ml.common.transformer_engine.evaluate, plus the selected model manifest.

Select models using tuning only. Freeze model, preprocessing and calibration before the one reserved final-test attempt. No final-test evaluation is launched by this notebook.

No datasets, dependencies or model weights are fetched. No model is trained by running all cells with the default flag. Fill only real, approved local paths. Do not store private review text in committed outputs. Source presence is not Phase 2/3 acceptance.

Read [the Phase 3 runbook](../documentation/phase3/RUNBOOK.md) first.

In [ ]:
from pathlib import Path
import sys

# Explicit opt-in for later local inspection; this delivery has not executed cells.
RUN_APPROVED_EXPERIMENTS = False
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / "ARCHITECTURE.md").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the D:/BDA_TA workspace.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Workspace:", ROOT)
print("Experiment execution enabled:", RUN_APPROVED_EXPERIMENTS)

## Later evaluator commands

```powershell
.\scripts\train_models.ps1 -Component sentiment_transformer -Action Evaluate -ConfigPath '<reviewed config>' -RunDirectory '<completed run>' -Split test -FinalTest -Device cpu
```

For TF-IDF use `-Component sentiment_tfidf` and its real config/run. Follow the component README for the frozen `selection.json` record. A failed reserved test attempt remains recorded; do not delete the reservation to rerun. Calibration-role diagnostics are distinguished from held-out performance.

In [ ]:
EVALUATION_REPORT = None  # Actual report path; never a manually filled metrics table.
if RUN_APPROVED_EXPERIMENTS and EVALUATION_REPORT:
    from ml.common.artifacts import read_json
    report = read_json(ROOT / EVALUATION_REPORT)
    assert report["split"] in {"tuning", "calibration", "test"}
    assert report["metrics"]["count"] > 0
    print({"task": report["task"], "split": report["split"],
           "metrics": report["metrics"], "cohorts": report.get("cohorts", {}),
           "manifest_hashes": report["manifest_hashes"]})
else:
    print("No actual report supplied; metrics and cohort coverage remain unavailable.")

In [ ]:
# Optional plot uses only an actual report; it never predicts or recomputes labels.
if RUN_APPROVED_EXPERIMENTS and EVALUATION_REPORT:
    import matplotlib.pyplot as plt
    matrix = report["metrics"]["confusion_matrix"]
    figure, axis = plt.subplots(figsize=(5, 4))
    image = axis.imshow(matrix, cmap="Blues")
    axis.set(xlabel="Predicted class index", ylabel="True class index",
             title=f"{report['task']} — {report['split']}")
    figure.colorbar(image, ax=axis)
    plt.show()
else:
    print("Plot unavailable until a real evaluation report exists.")

## Evaluation table to derive from reports

Report accuracy, macro/weighted F1, each class precision/recall/support, confusion matrix, NLL, multiclass Brier and 15-bin top-label ECE. Preserve the full task label space in cohort metrics and show zero class support. Overlapping cohorts are not independent samples.

Compare binary IMDb only against binary IMDb on identical hashes. Compare three-class movie models only on the same permitted labels, evaluation protocol and split fingerprints. Confidence calibration does not establish multilingual validity or sufficient audience evidence.

## Interpretation and acceptance

Record measured results, source/model licenses, exact config and manifest hashes, code revision, dependency versions, hardware and unsupported cohorts after the approved runs. Keep the final held-out set separate from model selection. No result has been measured in this notebook. Update the model card from actual reports; missing evidence remains missing.